In [9]:
# =========================================================
# STEP 1 — INVENTORY OF EXISTING WEATHER BATCHES
# =========================================================

from pathlib import Path
import pandas as pd

# ---------------------------------------------------------
# Project paths
# ---------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent

WEATHER_OUTPUT_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "open_meteo_daily_batches"
)

EXPECTED_DAYS = 2557
EXPECTED_BATCH_IDS = [1, 2, 3]

print("Project root:", PROJECT_ROOT)
print("Weather directory:", WEATHER_OUTPUT_DIR)
print("Directory exists:", WEATHER_OUTPUT_DIR.exists())


# ---------------------------------------------------------
# Inspect existing batch files
# ---------------------------------------------------------

inventory = []

for batch_id in EXPECTED_BATCH_IDS:

    weather_file = WEATHER_OUTPUT_DIR / (
        f"weather_batch_{batch_id:03d}.csv"
    )

    metadata_file = WEATHER_OUTPUT_DIR / (
        f"location_metadata_batch_{batch_id:03d}.csv"
    )

    record = {
        "batch_id": batch_id,
        "weather_exists": weather_file.exists(),
        "metadata_exists": metadata_file.exists(),
        "weather_size_mb": None,
        "metadata_size_mb": None,
        "weather_rows": None,
        "weather_columns": None,
        "metadata_rows": None,
        "metadata_columns": None,
        "read_error": None,
    }

    try:
        if weather_file.exists():
            record["weather_size_mb"] = round(
                weather_file.stat().st_size / (1024 ** 2), 3
            )

            weather_sample = pd.read_csv(
                weather_file,
                nrows=5
            )

            record["weather_columns"] = len(
                weather_sample.columns
            )

            record["weather_rows"] = sum(
                1
                for _ in open(
                    weather_file,
                    encoding="utf-8"
                )
            ) - 1

        if metadata_file.exists():
            record["metadata_size_mb"] = round(
                metadata_file.stat().st_size / (1024 ** 2), 3
            )

            metadata_sample = pd.read_csv(
                metadata_file,
                nrows=5
            )

            record["metadata_columns"] = len(
                metadata_sample.columns
            )

            record["metadata_rows"] = sum(
                1
                for _ in open(
                    metadata_file,
                    encoding="utf-8"
                )
            ) - 1

    except Exception as exc:
        record["read_error"] = str(exc)

    inventory.append(record)


# ---------------------------------------------------------
# Display inventory
# ---------------------------------------------------------

inventory_df = pd.DataFrame(inventory)

display(inventory_df)

print("\nExpected daily rows per batch:")
print("25 municipalities ×", EXPECTED_DAYS, "days =", 25 * EXPECTED_DAYS)

print("\nTotal expected rows for batches 1–3:")
print(3 * 25 * EXPECTED_DAYS)

Project root: c:\Users\Monica\Desktop\ETL\ODS2
Weather directory: c:\Users\Monica\Desktop\ETL\ODS2\data\raw\open_meteo_daily_batches
Directory exists: True


,batch_id,weather_exists,metadata_exists,weather_size_mb,metadata_size_mb,weather_rows,weather_columns,metadata_rows,metadata_columns,read_error
0,1,True,True,4.693,0.001,63925,12,25,7,None
1,2,True,True,4.733,0.001,63925,12,25,7,None
2,3,True,True,4.673,0.001,63925,12,25,7,None



Expected daily rows per batch:
25 municipalities × 2557 days = 63925

Total expected rows for batches 1–3:
191775


In [10]:
# =========================================================
# STEP 2 — SCHEMA AND DATA TYPE VALIDATION
# =========================================================

import pandas as pd
from pathlib import Path

batch_ids = [1, 2, 3]
schema_results = []
batch_schemas = {}
batch_samples = {}

for batch_id in batch_ids:

    weather_file = WEATHER_OUTPUT_DIR / (
        f"weather_batch_{batch_id:03d}.csv"
    )

    # Read a sample to inspect column names and inferred types
    sample = pd.read_csv(weather_file, nrows=1000)

    batch_samples[batch_id] = sample
    batch_schemas[batch_id] = list(sample.columns)

    for column in sample.columns:
        schema_results.append({
            "batch_id": batch_id,
            "column": column,
            "dtype": str(sample[column].dtype),
            "sample_nulls": int(sample[column].isna().sum()),
            "sample_rows": len(sample),
        })

schema_df = pd.DataFrame(schema_results)

print("COLUMN STRUCTURE AND TYPES")
display(schema_df)

# ---------------------------------------------------------
# Compare column names and order across batches
# ---------------------------------------------------------

reference_schema = batch_schemas[1]

schema_comparison = []

for batch_id, columns in batch_schemas.items():
    schema_comparison.append({
        "batch_id": batch_id,
        "column_count": len(columns),
        "same_columns_as_batch_1": set(columns) == set(reference_schema),
        "same_order_as_batch_1": columns == reference_schema,
        "missing_columns": [
            col for col in reference_schema if col not in columns
        ],
        "additional_columns": [
            col for col in columns if col not in reference_schema
        ],
    })

schema_comparison_df = pd.DataFrame(schema_comparison)

print("\nSCHEMA COMPARISON")
display(schema_comparison_df)

# ---------------------------------------------------------
# Inspect the actual column names
# ---------------------------------------------------------

print("\nCOLUMNS FOUND IN BATCH 1:")
for column in reference_schema:
    print("-", repr(column))

COLUMN STRUCTURE AND TYPES


,batch_id,column,dtype,sample_nulls,sample_rows
0,1,location_id,int64,0,1000
1,1,time,str,0,1000
2,1,temperature_2m_mean (°C),float64,0,1000
3,1,temperature_2m_max (°C),float64,0,1000
4,1,temperature_2m_min (°C),float64,0,1000
5,1,precipitation_sum (mm),float64,0,1000
6,1,relative_humidity_2m_mean (%),int64,0,1000
7,1,et0_fao_evapotranspiration (mm),float64,0,1000
8,1,Código Dane municipio,int64,0,1000
9,1,Municipio,str,0,1000



SCHEMA COMPARISON


,batch_id,column_count,same_columns_as_batch_1,same_order_as_batch_1,missing_columns,additional_columns
0,1,12,True,True,[],[]
1,2,12,True,True,[],[]
2,3,12,True,True,[],[]



COLUMNS FOUND IN BATCH 1:
- 'location_id'
- 'time'
- 'temperature_2m_mean (°C)'
- 'temperature_2m_max (°C)'
- 'temperature_2m_min (°C)'
- 'precipitation_sum (mm)'
- 'relative_humidity_2m_mean (%)'
- 'et0_fao_evapotranspiration (mm)'
- 'Código Dane municipio'
- 'Municipio'
- 'Latitud'
- 'longitud'


In [11]:

# =========================================================
# EVA-DIVIPOLA MUNICIPALITY MATCH DIAGNOSTICS
# =========================================================

import pandas as pd

EVA_CODE = "Código Dane municipio"
EVA_DEPT_CODE = "Código Dane departamento"
EVA_DEPT = "Departamento"
EVA_MUNICIPALITY = "Municipio"

DIV_CODE = "Código Municipio"
DIV_DEPT_CODE = "Código Departamento"
DIV_DEPT = "Nombre Departamento"
DIV_MUNICIPALITY = "Nombre Municipio"
DIV_TYPE = "Tipo: Municipio / Isla / Área no municipalizada"

# ---------------------------------------------------------
# 1. Normalize codes consistently
# ---------------------------------------------------------

def normalize_code(series):
    codes = series.astype("string").str.strip()
    codes = codes.str.replace(r"\.0$", "", regex=True)
    return codes.str.zfill(5).mask(codes.eq(""))

eva_geo = eva[
    [EVA_DEPT_CODE, EVA_DEPT, EVA_CODE, EVA_MUNICIPALITY]
].copy()

div_geo = divipola_work[
    [
        DIV_DEPT_CODE,
        DIV_DEPT,
        DIV_CODE,
        DIV_MUNICIPALITY,
        DIV_TYPE,
        "Latitud",
        "longitud",
    ]
].copy()

eva_geo["municipality_code"] = normalize_code(eva_geo[EVA_CODE])
div_geo["municipality_code"] = normalize_code(div_geo[DIV_CODE])

# Reject malformed non-null codes rather than silently matching them.
eva_bad_codes = eva_geo[
    eva_geo["municipality_code"].notna()
    & ~eva_geo["municipality_code"].str.fullmatch(r"\d{5}", na=False)
]

div_bad_codes = div_geo[
    div_geo["municipality_code"].notna()
    & ~div_geo["municipality_code"].str.fullmatch(r"\d{5}", na=False)
]

print("Malformed non-null EVA codes:", len(eva_bad_codes))
print("Malformed non-null DIVIPOLA codes:", len(div_bad_codes))

# ---------------------------------------------------------
# 2. Check duplicate and conflicting municipality codes
# ---------------------------------------------------------

eva_unique = eva_geo.dropna(subset=["municipality_code"]).drop_duplicates()

eva_conflicts = (
    eva_unique.groupby("municipality_code")
    .agg(
        municipality_names=(EVA_MUNICIPALITY, "nunique"),
        department_codes=(EVA_DEPT_CODE, "nunique"),
        department_names=(EVA_DEPT, "nunique"),
    )
    .query(
        "municipality_names > 1 "
        "or department_codes > 1 "
        "or department_names > 1"
    )
)

div_valid = div_geo.dropna(subset=["municipality_code"]).copy()

div_duplicate_codes = div_valid[
    div_valid["municipality_code"].duplicated(keep=False)
].sort_values("municipality_code")

print("\nDistinct EVA municipality codes:",
      eva_unique["municipality_code"].nunique())
print("Distinct DIVIPOLA codes:",
      div_valid["municipality_code"].nunique())
print("DIVIPOLA codes with duplicate rows:",
      div_duplicate_codes["municipality_code"].nunique())
print("EVA codes with conflicting municipality attributes:",
      len(eva_conflicts))

display(eva_conflicts.head(20))
display(div_duplicate_codes.head(20))

# Stop before joining if the reference key is ambiguous.
if div_valid["municipality_code"].duplicated().any():
    raise ValueError(
        "DIVIPOLA contains duplicate municipality codes. "
        "Inspect the division types before building the crosswalk."
    )

if not eva_conflicts.empty:
    raise ValueError(
        "EVA has conflicting attributes for some municipality codes. "
        "Resolve these before building a one-row-per-municipality catalog."
    )

# ---------------------------------------------------------
# 3. Create one EVA row per municipality and join by code
# ---------------------------------------------------------

eva_unique = (
    eva_unique.sort_values("municipality_code")
    .drop_duplicates("municipality_code")
)

crosswalk = eva_unique.merge(
    div_valid,
    on="municipality_code",
    how="left",
    validate="one_to_one",
    indicator=True,
)

print("\nEVA-DIVIPOLA match status:")
display(
    crosswalk["_merge"]
    .value_counts()
    .rename_axis("match_status")
    .reset_index(name="municipalities")
)

# ---------------------------------------------------------
# 4. Validate coordinates and municipality classification
# ---------------------------------------------------------

lat = pd.to_numeric(crosswalk["Latitud"], errors="coerce")
lon = pd.to_numeric(crosswalk["longitud"], errors="coerce")

crosswalk["coordinates_valid"] = (
    lat.between(-5, 14)
    & lon.between(-82, -66)
)

crosswalk["is_official_municipality"] = (
    crosswalk[DIV_TYPE].eq("Municipio")
)

crosswalk["eligible_for_weather"] = (
    crosswalk["_merge"].eq("both")
    & crosswalk["coordinates_valid"]
    & crosswalk["is_official_municipality"]
)

print("\nGeographic coverage summary:")
print("EVA municipalities:", len(crosswalk))
print("Matched to DIVIPOLA:", crosswalk["_merge"].eq("both").sum())
print("Unmatched to DIVIPOLA:", crosswalk["_merge"].eq("left_only").sum())
print("Invalid or missing coordinates:",
      (~crosswalk["coordinates_valid"]).sum())
print("Not classified as Municipality:",
      (~crosswalk["is_official_municipality"]).sum())
print("Eligible for weather extraction:",
      crosswalk["eligible_for_weather"].sum())

print("\nEVA municipalities requiring review:")
display(
    crosswalk.loc[
        ~crosswalk["eligible_for_weather"],
        [
            "municipality_code",
            EVA_MUNICIPALITY,
            EVA_DEPT,
            DIV_MUNICIPALITY,
            DIV_TYPE,
            "Latitud",
            "longitud",
            "_merge",
            "coordinates_valid",
        ],
    ]
)

# Keep the complete crosswalk for auditing.
# Do not replace municipality_locations until discrepancies are reviewed.


NameError: name 'eva' is not defined

In [ ]:
# =========================================================
# STEP 3 — FULL DATA QUALITY PROFILING
# =========================================================

import pandas as pd

BATCH_IDS = [1, 2, 3]
EXPECTED_DAYS = 2557
EXPECTED_START = pd.Timestamp("2019-01-01")
EXPECTED_END = pd.Timestamp("2025-12-31")

DATE_COL = "time"
MUNICIPALITY_CODE = "Código Dane municipio"

WEATHER_VARIABLES = [
    "temperature_2m_mean (°C)",
    "temperature_2m_max (°C)",
    "temperature_2m_min (°C)",
    "precipitation_sum (mm)",
    "relative_humidity_2m_mean (%)",
    "et0_fao_evapotranspiration (mm)",
]

# ---------------------------------------------------------
# 1. Load and consolidate the existing batches in memory
# ---------------------------------------------------------

frames = []

for batch_id in BATCH_IDS:
    file_path = WEATHER_OUTPUT_DIR / f"weather_batch_{batch_id:03d}.csv"

    batch_df = pd.read_csv(file_path)
    batch_df["source_batch_id"] = batch_id

    frames.append(batch_df)

weather_raw = pd.concat(frames, ignore_index=True)

print("CONSOLIDATED DATA")
print("Rows:", len(weather_raw))
print("Columns:", len(weather_raw.columns))
print("Municipalities:", weather_raw[MUNICIPALITY_CODE].nunique())
print("Date values as originally loaded:", weather_raw[DATE_COL].dtype)

# ---------------------------------------------------------
# 2. Missing values across all records
# ---------------------------------------------------------

null_summary = pd.DataFrame({
    "null_count": weather_raw.isna().sum(),
    "null_percentage": (
        weather_raw.isna().sum() / len(weather_raw) * 100
    ).round(4),
})

print("\nNULL VALUES — ALL RECORDS")
display(null_summary)

# ---------------------------------------------------------
# 3. Validate and convert dates in the working dataframe
# ---------------------------------------------------------

weather_raw[DATE_COL] = pd.to_datetime(
    weather_raw[DATE_COL],
    errors="coerce",
)

invalid_dates = weather_raw[DATE_COL].isna().sum()

outside_period = (
    weather_raw[DATE_COL].notna()
    & (
        (weather_raw[DATE_COL] < EXPECTED_START)
        | (weather_raw[DATE_COL] > EXPECTED_END)
    )
).sum()

print("\nDATE VALIDATION")
print("Invalid/unparseable dates:", int(invalid_dates))
print("Dates outside expected period:", int(outside_period))
print("Minimum valid date:", weather_raw[DATE_COL].min())
print("Maximum valid date:", weather_raw[DATE_COL].max())

# ---------------------------------------------------------
# 4. Coverage by source batch
# ---------------------------------------------------------

batch_summary = (
    weather_raw.groupby("source_batch_id")
    .agg(
        records=(DATE_COL, "size"),
        municipalities=(MUNICIPALITY_CODE, "nunique"),
        first_date=(DATE_COL, "min"),
        last_date=(DATE_COL, "max"),
        distinct_dates=(DATE_COL, "nunique"),
    )
    .reset_index()
)

batch_summary["expected_records"] = (
    batch_summary["municipalities"] * EXPECTED_DAYS
)
batch_summary["record_count_matches"] = (
    batch_summary["records"] == batch_summary["expected_records"]
)

print("\nBATCH COVERAGE")
display(batch_summary)

# ---------------------------------------------------------
# 5. Coverage for each municipality
# ---------------------------------------------------------

municipality_coverage = (
    weather_raw.groupby(MUNICIPALITY_CODE)
    .agg(
        municipality_name=("Municipio", "nunique"),
        records=(DATE_COL, "size"),
        distinct_dates=(DATE_COL, "nunique"),
        first_date=(DATE_COL, "min"),
        last_date=(DATE_COL, "max"),
        batches=("source_batch_id", "nunique"),
    )
    .reset_index()
)

municipality_coverage["expected_records"] = EXPECTED_DAYS
municipality_coverage["record_count_matches"] = (
    municipality_coverage["records"]
    == municipality_coverage["expected_records"]
)
municipality_coverage["date_count_matches"] = (
    municipality_coverage["distinct_dates"]
    == EXPECTED_DAYS
)
municipality_coverage["name_is_consistent"] = (
    municipality_coverage["municipality_name"] == 1
)

coverage_issues = municipality_coverage[
    ~(
        municipality_coverage["record_count_matches"]
        & municipality_coverage["date_count_matches"]
        & municipality_coverage["name_is_consistent"]
    )
]

print("\nMUNICIPALITY COVERAGE")
print("Total municipalities:", len(municipality_coverage))
print("Municipalities with coverage/name issues:", len(coverage_issues))
display(coverage_issues.head(20))

# ---------------------------------------------------------
# 6. Check uniqueness of the daily business key
# ---------------------------------------------------------

business_key = [MUNICIPALITY_CODE, DATE_COL]

duplicate_key_mask = weather_raw.duplicated(
    subset=business_key,
    keep=False,
)

duplicate_key_rows = weather_raw.loc[duplicate_key_mask].sort_values(
    business_key
)

print("\nDAILY BUSINESS KEY")
print("Rows participating in duplicate municipality-date keys:",
      int(duplicate_key_mask.sum()))
print("Distinct duplicated municipality-date keys:",
      int(weather_raw.loc[duplicate_key_mask, business_key]
          .drop_duplicates().shape[0]))

display(duplicate_key_rows.head(20))

# ---------------------------------------------------------
# 7. Basic meteorological range and consistency checks
# ---------------------------------------------------------

range_checks = {
    "temperature_mean_outside_-30_60": (
        (weather_raw["temperature_2m_mean (°C)"] < -30)
        | (weather_raw["temperature_2m_mean (°C)"] > 60)
    ),
    "temperature_max_below_min": (
        weather_raw["temperature_2m_max (°C)"]
        < weather_raw["temperature_2m_min (°C)"]
    ),
    "temperature_mean_above_max": (
        weather_raw["temperature_2m_mean (°C)"]
        > weather_raw["temperature_2m_max (°C)"]
    ),
    "temperature_mean_below_min": (
        weather_raw["temperature_2m_mean (°C)"]
        < weather_raw["temperature_2m_min (°C)"]
    ),
    "negative_precipitation": (
        weather_raw["precipitation_sum (mm)"] < 0
    ),
    "humidity_outside_0_100": (
        (weather_raw["relative_humidity_2m_mean (%)"] < 0)
        | (weather_raw["relative_humidity_2m_mean (%)"] > 100)
    ),
    "negative_et0": (
        weather_raw["et0_fao_evapotranspiration (mm)"] < 0
    ),
}

range_summary = pd.DataFrame([
    {
        "check": check_name,
        "affected_rows": int(mask.fillna(False).sum()),
    }
    for check_name, mask in range_checks.items()
])

print("\nBASIC METEOROLOGICAL CHECKS")
display(range_summary)

print("\nSTEP 3 COMPLETE — ORIGINAL CSV FILES WERE NOT MODIFIED")

CONSOLIDATED DATA
Rows: 191775
Columns: 13
Municipalities: 75
Date values as originally loaded: str

NULL VALUES — ALL RECORDS


,null_count,null_percentage
location_id,0,0.0
time,0,0.0
temperature_2m_mean (°C),0,0.0
temperature_2m_max (°C),0,0.0
temperature_2m_min (°C),0,0.0
precipitation_sum (mm),0,0.0
relative_humidity_2m_mean (%),0,0.0
et0_fao_evapotranspiration (mm),0,0.0
Código Dane municipio,0,0.0
Municipio,0,0.0



DATE VALIDATION
Invalid/unparseable dates: 0
Dates outside expected period: 0
Minimum valid date: 2019-01-01 00:00:00
Maximum valid date: 2025-12-31 00:00:00

BATCH COVERAGE


,source_batch_id,records,municipalities,first_date,last_date,distinct_dates,expected_records,record_count_matches
0,1,63925,25,2019-01-01,2025-12-31,2557,63925,True
1,2,63925,25,2019-01-01,2025-12-31,2557,63925,True
2,3,63925,25,2019-01-01,2025-12-31,2557,63925,True



MUNICIPALITY COVERAGE
Total municipalities: 75
Municipalities with coverage/name issues: 0


...


DAILY BUSINESS KEY
Rows participating in duplicate municipality-date keys: 0
Distinct duplicated municipality-date keys: 0


,location_id,time,temperature_2m_mean (°C),temperature_2m_max (°C),temperature_2m_min (°C),precipitation_sum (mm),relative_humidity_2m_mean (%),et0_fao_evapotranspiration (mm),Código Dane municipio,Municipio,Latitud,longitud,source_batch_id



BASIC METEOROLOGICAL CHECKS


,check,affected_rows
0,temperature_mean_outside_-30_60,0
1,temperature_max_below_min,0
2,temperature_mean_above_max,0
3,temperature_mean_below_min,0
4,negative_precipitation,0
5,humidity_outside_0_100,0
6,negative_et0,0



STEP 3 COMPLETE — ORIGINAL CSV FILES WERE NOT MODIFIED


In [ ]:
# =========================================================
# STEP 4 — MUNICIPALITY IDENTITY AND COORDINATE VALIDATION
# =========================================================

import pandas as pd

CODE = "Código Dane municipio"
NAME = "Municipio"
LAT = "Latitud"
LON = "longitud"

# ---------------------------------------------------------
# 1. Validate municipality names against DANE codes
# ---------------------------------------------------------

name_counts = (
    weather_raw.groupby(CODE)[NAME]
    .nunique(dropna=False)
)

inconsistent_names = name_counts[name_counts != 1]

print("MUNICIPALITY CODE ↔ NAME")
print("Municipality codes:", weather_raw[CODE].nunique())
print("Codes with inconsistent names:", len(inconsistent_names))

if len(inconsistent_names) > 0:
    display(
        weather_raw[
            weather_raw[CODE].isin(inconsistent_names.index)
        ][[CODE, NAME]]
        .drop_duplicates()
        .sort_values(CODE)
    )

# ---------------------------------------------------------
# 2. Validate coordinates for each municipality
# ---------------------------------------------------------

coordinate_counts = (
    weather_raw.groupby(CODE)
    .agg(
        distinct_latitudes=(LAT, "nunique"),
        distinct_longitudes=(LON, "nunique"),
    )
)

inconsistent_coordinates = coordinate_counts[
    (coordinate_counts["distinct_latitudes"] != 1)
    | (coordinate_counts["distinct_longitudes"] != 1)
]

print("\nMUNICIPALITY COORDINATES")
print("Codes with inconsistent coordinates:",
      len(inconsistent_coordinates))

if len(inconsistent_coordinates) > 0:
    display(inconsistent_coordinates)

# ---------------------------------------------------------
# 3. Validate location_id mapping within each batch
# ---------------------------------------------------------

location_mapping = (
    weather_raw.groupby(["source_batch_id", "location_id"])
    .agg(
        distinct_codes=(CODE, "nunique"),
        distinct_names=(NAME, "nunique"),
    )
    .reset_index()
)

invalid_location_ids = location_mapping[
    (location_mapping["distinct_codes"] != 1)
    | (location_mapping["distinct_names"] != 1)
]

print("\nLOCATION ID MAPPING")
print("Location IDs checked:", len(location_mapping))
print("Invalid location ID mappings:", len(invalid_location_ids))

if len(invalid_location_ids) > 0:
    display(invalid_location_ids)

# ---------------------------------------------------------
# 4. Inspect metadata files
# ---------------------------------------------------------

metadata_frames = []

for batch_id in BATCH_IDS:
    metadata_file = WEATHER_OUTPUT_DIR / (
        f"location_metadata_batch_{batch_id:03d}.csv"
    )

    metadata = pd.read_csv(metadata_file)
    metadata["source_batch_id"] = batch_id
    metadata_frames.append(metadata)

metadata_all = pd.concat(metadata_frames, ignore_index=True)

print("\nMETADATA SCHEMA")
print("Columns:", metadata_all.columns.tolist())
display(metadata_all.head(10))

print("\nMETADATA DATA TYPES")
display(
    pd.DataFrame({
        "column": metadata_all.columns,
        "dtype": [
            str(metadata_all[col].dtype)
            for col in metadata_all.columns
        ],
        "null_count": [
            int(metadata_all[col].isna().sum())
            for col in metadata_all.columns
        ],
    })
)

print("\nMETADATA ROW COUNTS")
display(
    metadata_all.groupby("source_batch_id")
    .size()
    .rename("metadata_rows")
    .reset_index()
)

print("\nSTEP 4 COMPLETE — NO SOURCE FILES MODIFIED")

MUNICIPALITY CODE ↔ NAME
Municipality codes: 75
Codes with inconsistent names: 0

MUNICIPALITY COORDINATES
Codes with inconsistent coordinates: 0

LOCATION ID MAPPING
Location IDs checked: 75
Invalid location ID mappings: 0

METADATA SCHEMA
Columns: ['location_id', 'latitude', 'longitude', 'elevation', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'source_batch_id']


,location_id,latitude,longitude,elevation,utc_offset_seconds,timezone,timezone_abbreviation,source_batch_id
0,0,6.221441,-75.551850,1475.0,-18000,America/Bogota,GMT-5,1
1,1,5.799649,-75.399350,2160.0,-18000,America/Bogota,GMT-5,1
2,2,6.713532,-75.996640,1915.0,-18000,America/Bogota,GMT-5,1
3,3,6.362039,-75.150760,1650.0,-18000,America/Bogota,GMT-5,1
4,4,6.010545,-75.738130,1379.0,-18000,America/Bogota,GMT-5,1
5,5,6.924429,-75.050600,1546.0,-18000,America/Bogota,GMT-5,1
6,6,5.729350,-75.860320,1274.0,-18000,America/Bogota,GMT-5,1
7,7,6.151143,-75.714294,1859.0,-18000,America/Bogota,GMT-5,1
8,8,6.854130,-75.290650,1696.0,-18000,America/Bogota,GMT-5,1
9,9,7.065026,-75.177370,1553.0,-18000,America/Bogota,GMT-5,1



METADATA DATA TYPES


,column,dtype,null_count
0,location_id,int64,0
1,latitude,float64,0
2,longitude,float64,0
3,elevation,float64,0
4,utc_offset_seconds,int64,0
5,timezone,str,0
6,timezone_abbreviation,str,0
7,source_batch_id,int64,0



METADATA ROW COUNTS


,source_batch_id,metadata_rows
0,1,25
1,2,25
2,3,25



STEP 4 COMPLETE — NO SOURCE FILES MODIFIED


In [ ]:
# =========================================================
# STEP 5 — METADATA INTEGRITY AND COORDINATE COMPARISON
# =========================================================

import numpy as np
import pandas as pd

CODE = "Código Dane municipio"
LAT = "Latitud"
LON = "longitud"

# ---------------------------------------------------------
# 1. Check uniqueness of metadata keys
# ---------------------------------------------------------

metadata_key = ["source_batch_id", "location_id"]

duplicate_metadata = metadata_all.duplicated(
    subset=metadata_key,
    keep=False,
)

print("METADATA KEY")
print("Metadata rows:", len(metadata_all))
print(
    "Duplicate metadata key rows:",
    int(duplicate_metadata.sum())
)

if duplicate_metadata.any():
    display(
        metadata_all.loc[duplicate_metadata]
        .sort_values(metadata_key)
    )

# ---------------------------------------------------------
# 2. Match daily records to metadata
# ---------------------------------------------------------

weather_locations = (
    weather_raw[
        ["source_batch_id", "location_id", CODE, "Municipio", LAT, LON]
    ]
    .drop_duplicates()
)

metadata_match = weather_locations.merge(
    metadata_all,
    on=["source_batch_id", "location_id"],
    how="left",
    validate="many_to_one",
    indicator=True,
)

unmatched_locations = metadata_match[
    metadata_match["_merge"] != "both"
]

print("\nDAILY DATA ↔ METADATA")
print("Municipality-location combinations:", len(weather_locations))
print("Unmatched combinations:", len(unmatched_locations))

if len(unmatched_locations) > 0:
    display(unmatched_locations)

# ---------------------------------------------------------
# 3. Check timezone and UTC offset
# ---------------------------------------------------------

print("\nTIMEZONE VALUES")
display(
    metadata_all.groupby(
        ["timezone", "utc_offset_seconds", "timezone_abbreviation"],
        dropna=False,
    )
    .size()
    .rename("locations")
    .reset_index()
)

# ---------------------------------------------------------
# 4. Compare input coordinates with API-returned coordinates
# ---------------------------------------------------------

# Haversine distance in kilometers
lat1 = np.radians(metadata_match[LAT])
lon1 = np.radians(metadata_match[LON])
lat2 = np.radians(metadata_match["latitude"])
lon2 = np.radians(metadata_match["longitude"])

delta_lat = lat2 - lat1
delta_lon = lon2 - lon1

a = (
    np.sin(delta_lat / 2) ** 2
    + np.cos(lat1) * np.cos(lat2) * np.sin(delta_lon / 2) ** 2
)

a = a.clip(0, 1)

metadata_match["coordinate_difference_km"] = (
    6371 * 2 * np.arcsin(np.sqrt(a))
)

coordinate_summary = (
    metadata_match.groupby("source_batch_id")
    ["coordinate_difference_km"]
    .agg(
        min_km="min",
        median_km="median",
        max_km="max",
    )
    .round(3)
    .reset_index()
)

print("\nCOORDINATE DIFFERENCES")
display(coordinate_summary)

print("\nLargest coordinate differences:")
display(
    metadata_match.nlargest(
        10, "coordinate_difference_km"
    )[
        [
            "source_batch_id",
            "location_id",
            CODE,
            "Municipio",
            LAT,
            LON,
            "latitude",
            "longitude",
            "coordinate_difference_km",
        ]
    ]
)

print("\nSTEP 5 COMPLETE — NO SOURCE FILES MODIFIED")

METADATA KEY
Metadata rows: 75
Duplicate metadata key rows: 0

DAILY DATA ↔ METADATA
Municipality-location combinations: 75
Unmatched combinations: 0

TIMEZONE VALUES


,timezone,utc_offset_seconds,timezone_abbreviation,locations
0,America/Bogota,-18000,GMT-5,75



COORDINATE DIFFERENCES


,source_batch_id,min_km,median_km,max_km
0,1,1.736,4.617,11.725
1,2,1.005,5.000,15.419
2,3,0.998,4.173,12.671



Largest coordinate differences:


,source_batch_id,location_id,Código Dane municipio,Municipio,Latitud,longitud,latitude,longitude,coordinate_difference_km
28,2,3,5138,Cañasgordas,6.753859,-76.028228,6.854130,-76.124695,15.419492
60,3,10,5364,Jardín,5.597542,-75.818982,5.659051,-75.722595,12.670587
66,3,16,5411,Liborina,6.677316,-75.812838,6.572935,-75.793430,11.802937
2,1,2,5004,Abriaquí,6.632282,-76.064304,6.713532,-75.996640,11.724725
58,3,8,5360,Itagüí,6.175079,-75.612056,6.151143,-75.714294,11.611798
26,2,1,5129,Caldas,6.091077,-75.633673,6.010545,-75.663055,9.525916
16,1,16,5079,Barbosa,6.439195,-75.331627,6.502636,-75.276855,9.294380
47,2,22,5282,Fredonia,5.928039,-75.675072,6.010545,-75.663055,9.270007
48,2,23,5284,Frontino,6.776066,-76.130765,6.854130,-76.124695,8.706187
63,3,13,5380,La Estrella,6.145238,-75.637708,6.151143,-75.714294,8.492411



STEP 5 COMPLETE — NO SOURCE FILES MODIFIED


In [ ]:
# =========================================================
# STEP 6 — DANE CODE FORMAT AND REFERENCE DISCOVERY
# =========================================================

# ---------------------------------------------------------
# 1. Inspect municipality code format
# ---------------------------------------------------------

code_as_text = weather_raw["Código Dane municipio"].astype("string").str.strip()

code_format = pd.DataFrame({
    "code_as_stored": code_as_text.drop_duplicates().sort_values().tolist()
})

code_format["length"] = code_format["code_as_stored"].str.len()
code_format["is_numeric"] = code_format["code_as_stored"].str.fullmatch(
    r"\d+"
)

print("MUNICIPALITY CODE FORMAT")
display(
    code_format.groupby(
        ["length", "is_numeric"],
        dropna=False
    )
    .size()
    .rename("distinct_codes")
    .reset_index()
)

print("\nEXAMPLE CODES AS CURRENTLY STORED")
display(code_format.head(20))

print("\nCODES THAT DO NOT HAVE FIVE CHARACTERS")
display(code_format[code_format["length"] != 5].head(30))

# ---------------------------------------------------------
# 2. Discover likely DIVIPOLA reference DataFrames
# ---------------------------------------------------------

print("\nPOSSIBLE REFERENCE DATAFRAMES ALREADY LOADED")

reference_candidates = []

for variable_name, variable_value in globals().copy().items():
    if isinstance(variable_value, pd.DataFrame):
        columns_text = " ".join(
            str(column).lower()
            for column in variable_value.columns
        )

        name_text = variable_name.lower()

        if (
            "divipola" in name_text
            or (
                "municipio" in columns_text
                and any(term in columns_text for term in [
                    "código dane municipio",
                    "codigo dane municipio",
                    "codigo_municipio",
                    "cod_mpio",
                    "código municipio",
                    "codigo municipio",
                ])
            )
        ):
            reference_candidates.append({
                "variable_name": variable_name,
                "rows": len(variable_value),
                "columns": list(variable_value.columns),
            })

display(pd.DataFrame(reference_candidates))

print("\nSTEP 6 COMPLETE — NO SOURCE FILES MODIFIED")

MUNICIPALITY CODE FORMAT


,length,is_numeric,distinct_codes
0,4,True,75



EXAMPLE CODES AS CURRENTLY STORED


,code_as_stored,length,is_numeric
0,5001,4,True
1,5002,4,True
2,5004,4,True
3,5021,4,True
4,5030,4,True
5,5031,4,True
6,5034,4,True
7,5036,4,True
8,5038,4,True
9,5040,4,True



CODES THAT DO NOT HAVE FIVE CHARACTERS


,code_as_stored,length,is_numeric
0,5001,4,True
1,5002,4,True
2,5004,4,True
3,5021,4,True
4,5030,4,True
5,5031,4,True
6,5034,4,True
7,5036,4,True
8,5038,4,True
9,5040,4,True



POSSIBLE REFERENCE DATAFRAMES ALREADY LOADED


,variable_name,rows,columns
0,weather_sample,5,"[location_id, time, temperature_2m_mean (°C), ..."
1,sample,1000,"[location_id, time, temperature_2m_mean (°C), ..."
2,batch_df,63925,"[location_id, time, temperature_2m_mean (°C), ..."
3,weather_raw,191775,"[location_id, time, temperature_2m_mean (°C), ..."
4,municipality_coverage,75,"[Código Dane municipio, municipality_name, rec..."
5,coverage_issues,0,"[Código Dane municipio, municipality_name, rec..."
6,duplicate_key_rows,0,"[location_id, time, temperature_2m_mean (°C), ..."
7,weather_locations,75,"[source_batch_id, location_id, Código Dane mun..."
8,metadata_match,75,"[source_batch_id, location_id, Código Dane mun..."
9,unmatched_locations,0,"[source_batch_id, location_id, Código Dane mun..."



STEP 6 COMPLETE — NO SOURCE FILES MODIFIED


In [ ]:
# =========================================================
# STEP 7 — NORMALIZE MUNICIPALITY CODES AND FIND DIVIPOLA
# =========================================================

from pathlib import Path
import pandas as pd

CODE = "Código Dane municipio"

# ---------------------------------------------------------
# 1. Preserve original codes and create normalized codes
# ---------------------------------------------------------

weather_raw["codigo_dane_municipio"] = (
    weather_raw[CODE]
    .astype("string")
    .str.strip()
    .str.zfill(5)
)

print("CODE NORMALIZATION")
display(
    weather_raw[[CODE, "codigo_dane_municipio"]]
    .drop_duplicates()
    .sort_values("codigo_dane_municipio")
    .head(20)
)

normalized_codes = weather_raw["codigo_dane_municipio"]

print("\nDistinct normalized codes:", normalized_codes.nunique())
print(
    "Codes with exactly five digits:",
    int(normalized_codes.str.fullmatch(r"\d{5}").sum())
)
print(
    "Rows with invalid normalized format:",
    int((~normalized_codes.str.fullmatch(r"\d{5}")).sum())
)

# ---------------------------------------------------------
# 2. Search likely reference files in the project
# ---------------------------------------------------------

project_root = Path(PROJECT_ROOT)

reference_extensions = {".csv", ".xlsx", ".xls", ".parquet"}
reference_files = []

for path in project_root.rglob("*"):
    if not path.is_file():
        continue

    if path.suffix.lower() not in reference_extensions:
        continue

    if any(part.startswith(".") for part in path.parts):
        continue

    filename = path.name.lower()

    if any(term in filename for term in [
        "divipola",
        "municipio",
        "municipal",
        "dane",
    ]):
        reference_files.append({
            "file_name": path.name,
            "extension": path.suffix.lower(),
            "path": str(path),
            "size_mb": round(path.stat().st_size / (1024 ** 2), 3),
        })

print("\nPOSSIBLE DIVIPOLA REFERENCE FILES")
display(pd.DataFrame(reference_files))

print("\nSTEP 7 COMPLETE — ORIGINAL CSV FILES NOT MODIFIED")

CODE NORMALIZATION


,Código Dane municipio,codigo_dane_municipio
0,5001,05001
2557,5002,05002
5114,5004,05004
7671,5021,05021
10228,5030,05030
12785,5031,05031
15342,5034,05034
17899,5036,05036
20456,5038,05038
23013,5040,05040



Distinct normalized codes: 75
Codes with exactly five digits: 191775
Rows with invalid normalized format: 0

POSSIBLE DIVIPOLA REFERENCE FILES


,file_name,extension,path,size_mb
0,municipios_codigo_multiples_nombres.csv,.csv,c:\Users\Monica\Desktop\ETL\ODS2\outputs\profi...,0.000
1,municipios_nombre_multiples_codigos.csv,.csv,c:\Users\Monica\Desktop\ETL\ODS2\outputs\profi...,0.002
2,municipios_relaciones_inconsistentes.csv,.csv,c:\Users\Monica\Desktop\ETL\ODS2\outputs\profi...,0.000
3,divipola.csv,.csv,c:\Users\Monica\Desktop\ETL\ODS2\data\raw\divi...,0.078
4,Evaluaciones_Agropecuarias_Municipales_–_EVA._...,.csv,c:\Users\Monica\Desktop\ETL\ODS2\data\raw\Eval...,31.972



STEP 7 COMPLETE — ORIGINAL CSV FILES NOT MODIFIED


In [ ]:
from sqlalchemy import text

engine = crear_engine(DATABASE_NAME)

consultas = {
    "dim_geografia": "SELECT COUNT(*) FROM dim_geografia",
    "dim_cultivo": "SELECT COUNT(*) FROM dim_cultivo",
    "dim_tiempo": "SELECT COUNT(*) FROM dim_tiempo",
    "fact_produccion_agricola": (
        "SELECT COUNT(*) FROM fact_produccion_agricola"
    ),
}

with engine.connect() as connection:
    for tabla, consulta in consultas.items():
        cantidad = connection.execute(
            text(consulta)
        ).scalar_one()

        print(f"{tabla}: {cantidad:,} filas")

engine.dispose()

NameError: name 'crear_engine' is not defined